# KRX 종목 중 Google 뉴스 언급 빈도 상위 N종목 추출

한국거래소(KRX) 상장 종목 중 Google 뉴스에서 가장 빈번하게 거론되는 종목을 추출합니다.

**동작 방식**
1. KRX 전체 상장 종목 리스트(종목명, 티커)를 가져온다. (`FinanceDataReader` 사용)
2. 여러 검색 키워드로 Google 뉴스 RSS를 조회하되, 검색어에 `after:YYYY-MM-DD before:YYYY-MM-DD` 연산자를 붙여 지정한 기간의 기사만 가져온다. (RSS의 실제 발행일로 한 번 더 필터링)
3. 수집한 기사 텍스트 안에서 각 종목명이 몇 번 등장하는지 카운트한다.
4. 언급 횟수 기준으로 내림차순 정렬하여 상위 N종목을 산출한다.
5. 결과를 엑셀(.xlsx) 파일로 저장한다.

**주의**
- 이 방식은 "구글 트렌드/토픽"의 공식 순위 API가 아니라, 구글 뉴스 검색 결과를 기반으로 한 근사치입니다.
- 종목명이 너무 짧으면(예: "한화", "SK" 등) 일반명사와 겹쳐 오탐이 생길 수 있으므로 `MIN_NAME_LEN` 값으로 필터링합니다.
- 네트워크 호출량이 많으므로 요청 사이에 딜레이를 둡니다.

## 0. 패키지 설치
필요 시 아래 셀을 한 번 실행하세요.

In [ ]:
%pip install FinanceDataReader feedparser requests pandas openpyxl --quiet

## 1. 라이브러리 임포트

In [4]:
import re
import time
from datetime import datetime, date
from collections import Counter

import requests
import feedparser
import pandas as pd
import FinanceDataReader as fdr

## 2. 설정값

아래 변수들을 원하는 값으로 수정한 뒤 실행하세요.

- `START_DATE`, `END_DATE`: 조회 기간 (`YYYY-MM-DD` 형식 문자열, `None`이면 기간 제한 없음)
- `TOP_N`: 추출할 종목 수
- `MIN_NAME_LEN`: 오탐 방지를 위한 최소 종목명 길이
- `OUTPUT_PATH`: 결과를 저장할 엑셀 파일 경로

In [5]:
# ------------------------------------------------------------------
# 설정값 - 필요에 따라 수정하세요
# ------------------------------------------------------------------
START_DATE = "2026-06-01"   # 조회 시작일 (YYYY-MM-DD) 또는 None
END_DATE = "2026-06-30"     # 조회 종료일 (YYYY-MM-DD) 또는 None
TOP_N = 20                  # 추출할 종목 수
MIN_NAME_LEN = 2            # 오탐 방지를 위한 최소 종목명 길이
OUTPUT_PATH = "krx_google_trending_top20.xlsx"  # 결과 저장 경로

REQUEST_DELAY = 1.0            # 요청 간 딜레이(초) - 과도한 요청 방지
MAX_ARTICLES_PER_QUERY = 100   # 쿼리당 최대 수집 기사 수 (RSS 특성상 대략치)

# Google 뉴스에서 시장 전반의 기사를 폭넓게 긁어오기 위한 검색어들
SEARCH_QUERIES = [
    "코스피",
    "코스닥",
    "증시",
    "주식시장",
    "상한가",
    "실적발표",
    "주가",
]

# 날짜 형식 간단 검증
for label, value in (("START_DATE", START_DATE), ("END_DATE", END_DATE)):
    if value:
        datetime.strptime(value, "%Y-%m-%d")  # 형식이 틀리면 여기서 에러 발생

if START_DATE and END_DATE and START_DATE > END_DATE:
    raise ValueError("START_DATE는 END_DATE보다 이전이어야 합니다.")

## 3. KRX 상장 종목 리스트 수집

In [6]:
def get_krx_stock_list(min_name_len):
    """KRX 전체 상장 종목(코스피+코스닥) 목록을 가져온다."""
    print("[1/4] KRX 상장 종목 리스트 수집 중...")
    df = fdr.StockListing("KRX")  # 종목명, 코드 등 포함
    name_col = "Name" if "Name" in df.columns else "name"
    code_col = "Code" if "Code" in df.columns else "code"

    stocks = df[[code_col, name_col]].dropna()
    stocks = stocks[stocks[name_col].str.len() >= min_name_len]
    stock_list = list(zip(stocks[code_col], stocks[name_col]))
    print(f"  -> 총 {len(stock_list)}개 종목 확보")
    return stock_list


stock_list = get_krx_stock_list(MIN_NAME_LEN)
stock_list[:10]  # 확인용 미리보기

[1/4] KRX 상장 종목 리스트 수집 중...
  -> 총 2873개 종목 확보


[('005930', '삼성전자'),
 ('000660', 'SK하이닉스'),
 ('402340', 'SK스퀘어'),
 ('005935', '삼성전자우'),
 ('009150', '삼성전기'),
 ('005380', '현대차'),
 ('373220', 'LG에너지솔루션'),
 ('032830', '삼성생명'),
 ('028260', '삼성물산'),
 ('207940', '삼성바이오로직스')]

## 4. Google 뉴스 기사 수집 (기간 필터 적용)

In [7]:
def build_query(base_query, start, end):
    """검색어에 기간 필터(after/before)를 덧붙인다."""
    q = base_query
    if start:
        q += f" after:{start}"
    if end:
        q += f" before:{end}"
    return q


def parse_entry_date(entry):
    """RSS 엔트리에서 발행일을 date 객체로 변환한다. 실패 시 None."""
    parsed = getattr(entry, "published_parsed", None)
    if not parsed:
        return None
    try:
        return date(parsed.tm_year, parsed.tm_mon, parsed.tm_mday)
    except Exception:
        return None


def fetch_google_news_titles(query, start, end, lang="ko", country="KR"):
    """Google 뉴스 RSS에서 특정 검색어(+기간)에 대한 기사 제목+요약 리스트를 가져온다."""
    url = (
        f"https://news.google.com/rss/search?q={requests.utils.quote(query)}"
        f"&hl={lang}&gl={country}&ceid={country}:{lang}"
    )
    try:
        resp = requests.get(url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        resp.raise_for_status()
        feed = feedparser.parse(resp.content)
    except Exception as e:
        print(f"  [경고] '{query}' 요청 실패: {e}")
        return []

    start_d = datetime.strptime(start, "%Y-%m-%d").date() if start else None
    end_d = datetime.strptime(end, "%Y-%m-%d").date() if end else None

    texts = []
    for entry in feed.entries[:MAX_ARTICLES_PER_QUERY]:
        entry_date = parse_entry_date(entry)
        if entry_date:
            if start_d and entry_date < start_d:
                continue
            if end_d and entry_date > end_d:
                continue

        title = getattr(entry, "title", "") or ""
        summary = getattr(entry, "summary", "") or ""
        texts.append(f"{title} {summary}")
    return texts


def collect_all_articles(start, end):
    """설정된 검색어(+기간)들로 뉴스 기사 텍스트를 모두 수집한다."""
    print("[2/4] Google 뉴스 기사 수집 중...")
    if start or end:
        print(f"  - 조회 기간: {start or '제한없음'} ~ {end or '제한없음'}")

    all_texts = []
    for base_q in SEARCH_QUERIES:
        query = build_query(base_q, start, end)
        print(f"  - 검색어: {query}")
        texts = fetch_google_news_titles(query, start, end)
        all_texts.extend(texts)
        time.sleep(REQUEST_DELAY)
    print(f"  -> 총 {len(all_texts)}개 기사(제목+요약) 수집")
    return all_texts


articles = collect_all_articles(START_DATE, END_DATE)
len(articles)

[2/4] Google 뉴스 기사 수집 중...
  - 조회 기간: 2026-06-01 ~ 2026-06-30
  - 검색어: 코스피 after:2026-06-01 before:2026-06-30
  - 검색어: 코스닥 after:2026-06-01 before:2026-06-30
  - 검색어: 증시 after:2026-06-01 before:2026-06-30
  - 검색어: 주식시장 after:2026-06-01 before:2026-06-30
  - 검색어: 상한가 after:2026-06-01 before:2026-06-30
  - 검색어: 실적발표 after:2026-06-01 before:2026-06-30
  - 검색어: 주가 after:2026-06-01 before:2026-06-30
  -> 총 700개 기사(제목+요약) 수집


700

## 5. 종목명 언급 빈도 집계

In [8]:
def count_stock_mentions(stock_list, articles):
    """기사 텍스트 안에서 각 종목명 언급 횟수를 카운트한다."""
    print("[3/4] 종목명 언급 빈도 집계 중...")
    counter = Counter()

    # 긴 종목명부터 매칭해서 짧은 이름이 긴 이름의 부분 문자열로 오매칭되는 것을 최소화
    sorted_stocks = sorted(stock_list, key=lambda x: len(x[1]), reverse=True)
    combined_text = "\n".join(articles)

    for code, name in sorted_stocks:
        pattern = re.escape(name)
        count = len(re.findall(pattern, combined_text))
        if count > 0:
            counter[(code, name)] = count

    return counter


if not articles:
    raise RuntimeError("수집된 기사가 없습니다. 네트워크 연결, 검색어, 또는 조회 기간을 확인하세요.")

counter = count_stock_mentions(stock_list, articles)
top_stocks = counter.most_common(TOP_N)

if not top_stocks:
    raise RuntimeError("언급된 종목을 찾지 못했습니다.")

result_df = pd.DataFrame(
    [
        {"순위": rank, "종목명": name, "종목코드": code, "언급 횟수": count}
        for rank, ((code, name), count) in enumerate(top_stocks, start=1)
    ]
)
result_df

[3/4] 종목명 언급 빈도 집계 중...


,순위,종목명,종목코드,언급 횟수
0,1,E1,017940,289
1,2,YW,051390,132
2,3,SG,255220,86
3,4,DB,012030,85
4,5,SK,034730,76
5,6,이닉스,452400,66
6,7,YTN,040300,65
7,8,3S,060310,62
8,9,E8,418620,51
9,10,SK하이닉스,000660,48


## 6. 결과를 엑셀 파일로 저장

In [ ]:
def save_to_excel(df, output_path, start, end):
    """결과를 엑셀 파일로 저장한다."""
    print(f"[4/4] 엑셀 파일 저장 중... ({output_path})")

    with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
        df.to_excel(writer, index=False, sheet_name="결과")

        meta = pd.DataFrame(
            {
                "항목": ["조회 시작일", "조회 종료일", "생성 시각"],
                "값": [
                    start or "제한없음",
                    end or "제한없음",
                    datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
                ],
            }
        )
        meta.to_excel(writer, index=False, sheet_name="조회조건")

        worksheet = writer.sheets["결과"]
        for i, col in enumerate(df.columns, start=1):
            max_len = max(df[col].astype(str).map(len).max(), len(col)) + 2
            worksheet.column_dimensions[chr(64 + i)].width = max_len

    print(f"  -> 저장 완료: {output_path}")


save_to_excel(result_df, OUTPUT_PATH, START_DATE, END_DATE)

In [ ]:
!pip install pytrends

In [ ]:
import time
import pandas as pd
from pykrx import stock
from pytrends.request import TrendReq

def get_google_trends_top20():
    # 1. KRX에서 시가총액 상위 100개 종목 추출 (전체 종목 검색은 시간/제한 초과)
    print("[1/3] KRX에서 시가총액 상위 100개 종목을 추출 중입니다...")
    # 최근 영업일 기준 (오늘 날짜로 해도 pykrx가 자동으로 최근 영업일을 잡아줍니다)
    df_krx = stock.get_market_cap()
    df_top100 = df_krx.sort_values(by='시가총액', ascending=False).head(100)
    
    tickers = df_top100.index.tolist()
    stock_names = [stock.get_market_ticker_name(ticker) for ticker in tickers]
    
    # 티커와 종목명 매칭 딕셔너리
    name_to_ticker = dict(zip(stock_names, tickers))

    # 2. 구글 트렌드 설정
    print("[2/3] 구글 트렌드에서 데이터를 수집 중입니다. (약 1~2분 소장)")
    pytrends = TrendReq(hl='ko', tz=540, timeout=(10, 25))
    
    results = {}
    batch_size = 5 # 구글 트렌드는 한 번에 5개까지만 검색 가능

    # 5개씩 묶어서 배치 처리
    for i in range(0, len(stock_names), batch_size):
        batch = stock_names[i:i + batch_size]
        try:
            # 최근 7일간 데이터 수집 (기간은 조정 가능: 'today 1-m' 등)
            pytrends.build_payload(batch, timeframe='today 7-d')
            interest_over_time = pytrends.interest_over_time()

            if not interest_over_time.empty:
                # 구글 트렌드는 0~100의 상대적 수치를 반환함
                # 평균 검색 관심도를 계산하여 저장
                for col in batch:
                    if col in interest_over_time.columns:
                        results[col] = interest_over_time[col].mean()
            
            # IP 차단 방지를 위해 필수적인 대기 시간
            time.sleep(2) 
            
        except Exception as e:
            print(f"에러 발생 ({batch}): {e}")
            time.sleep(5) # 에러 발생 시 좀 더 긴 대기

    # 3. 결과 정렬 및 Top 20 추출
    print("[3/3] 데이터 정렬 중...")
    # 관심도가 높은 순으로 정렬
    sorted_results = sorted(results.items(), key=lambda x: x[1], reverse=True)[:20]
    
    # 데이터프레임으로 변환
    df_result = pd.DataFrame(sorted_results, columns=['종목명', '구글 트렌드 평균 관심도'])
    
    # 티커(종목코드) 추가
    df_result['종목코드'] = df_result['종목명'].map(name_to_ticker)
    df_result = df_result[['종목코드', '종목명', '구글 트렌드 평균 관심도']]
    
    # 순위 컬럼 추가
    df_result.index = range(1, len(df_result) + 1)
    df_result.index.name = '순위'

    return df_result

# 실행
if __name__ == "__main__":
    top20_df = get_google_trends_top20()
    print("\n" + "="*50)
    print("🚀 구글 트렌드 가장 많이 거론된 KRX 상위 20종목")
    print("="*50)
    print(top20_df)

In [9]:
import FinanceDataReader as fdr
from pytrends.request import TrendReq
import pandas as pd
import time
from datetime import datetime, timedelta

def get_krx_stock_names():
    """KRX 상장 종목 리스트 가져오기"""
    print("KRX 상장 종목 리스트를 가져오는 중...")
    kospi = fdr.StockListing('KOSPI')
    kosdaq = fdr.StockListing('KOSDAQ')
    
    stocks = pd.concat([kospi[['Code', 'Name']], kosdaq[['Code', 'Name']]])
    stocks = stocks.drop_duplicates(subset='Code')
    
    stock_names = stocks['Name'].unique().tolist()
    # 오매칭 방지를 위해 2글자 이상인 종목명만 리스트화
    stock_names = [name for name in stock_names if len(name) >= 2]
    
    return stocks, stock_names

def get_google_trends_related_queries(start_date, end_date):
    """구글 트렌드 연관 검색어 가져오기 (재시도 로직 및 API 요청 최소화 적용)"""
    print(f"구글 트렌드 데이터를 가져오는 중... (기간: {start_date} ~ {end_date})")
    
    # 구글 트렌드에서 검색할 핵심 키워드 5개
    # pytrends는 최대 5개 키워드까지 한 번에 조회할 수 있음 (API 요청을 5번에서 1번으로 줄여 429 에러 방지)
    kw_list = ["주식", "코스피", "코스닥", "급등주", "주식 추천"]
    
    max_retries = 3
    retry_delay = 15  # 429 에러 발생 시 15초 대기 후 재시도
    
    for attempt in range(max_retries):
        try:
            pytrends = TrendReq(hl='ko', tz=540)
            timeframe_str = f"{start_date} {end_date}"
            
            # 5개 키워드를 한 번에 요청
            pytrends.build_payload(kw_list, cat=0, timeframe=timeframe_str, geo='KR', gprop='')
            related_queries = pytrends.related_queries()
            
            all_queries = []
            for kw in kw_list:
                if kw in related_queries:
                    top_df = related_queries[kw]['top']
                    if top_df is not None:
                        all_queries.append(top_df)
            
            if all_queries:
                result_df = pd.concat(all_queries).drop_duplicates(subset='query')
                print(f"✅ 트렌드 연관 검색어 {len(result_df)}개 추출 성공.")
                return result_df
            else:
                print("추출된 연관 검색어가 없습니다. 기간을 다시 확인해주세요.")
                return pd.DataFrame()
                
        except Exception as e:
            print(f"⚠️ 구글 트렌드 조회 중 에러 발생 (시도 {attempt + 1}/{max_retries}): {e}")
            if attempt < max_retries - 1:
                print(f"{retry_delay}초 대기 후 재시도합니다...")
                time.sleep(retry_delay)
            else:
                print("❌ 최대 재시도 횟수를 초과했습니다. 잠시 후 다시 실행해주세요.")
                return pd.DataFrame()

def find_top_20_stocks(start_date, end_date):
    """메인 실행 함수"""
    krx_stocks, stock_names = get_krx_stock_names()
    trends_df = get_google_trends_related_queries(start_date, end_date)
    
    if trends_df.empty:
        print("구글 트렌드 데이터가 없어 종목 매칭을 진행할 수 없습니다.")
        return pd.DataFrame()
        
    print("KRX 종목과 구글 트렌드 검색어를 매칭하는 중...")
    
    stock_scores = {}
    sorted_stock_names = sorted(stock_names, key=len, reverse=True)
    
    for idx, row in trends_df.iterrows():
        query = row['query']
        value = row['value']
        
        for name in sorted_stock_names:
            if name in query:
                if name in stock_scores:
                    stock_scores[name] += value
                else:
                    stock_scores[name] = value
                break 
                
    ranked_stocks = pd.DataFrame(
        list(stock_scores.items()), columns=['Name', 'Trend_Score']
    ).sort_values(by='Trend_Score', ascending=False).head(20)
    
    result = pd.merge(ranked_stocks, krx_stocks[['Name', 'Code']], on='Name', how='left')
    return result

In [12]:
# ==========================================
# 🔽 조회할 기간(시작일, 종료일)을 설정하세요. (YYYY-MM-DD 형식)
# 과거 날짜만 입력 가능합니다. 미래 날짜 입력 시 자동으로 최근 1개월로 보정됩니다.
# ==========================================
start_date = "2025-05-01"
end_date = "2025-05-31"
# ==========================================

today_str = datetime.today().strftime("%Y-%m-%d")

# 미래 날짜 입력 방지 로직
if start_date > today_str or end_date > today_str:
    print("🚫 미래 날짜의 데이터는 조회할 수 없습니다.")
    print("👉 조회 기간을 '최근 1개월'로 자동 변경하여 진행합니다.\n")
    end_date = today_str
    start_date = (datetime.today() - timedelta(days=30)).strftime("%Y-%m-%d")

# 날짜 형식 검사
try:
    datetime.strptime(start_date, "%Y-%m-%d")
    datetime.strptime(end_date, "%Y-%m-%d")
except ValueError:
    print("❌ 날짜 형식이 잘못되었습니다. 'YYYY-MM-DD' 형식으로 입력해주세요.")
else:
    # 함수 실행
    top_20_result = find_top_20_stocks(start_date, end_date)
    
    # 결과 출력
    if not top_20_result.empty:
        print("\n" + "="*60)
        print(f"🏆 구글 트렌드 최다 언급 KRX 상위 20종목 ({start_date} ~ {end_date})")
        print("="*60)
        
        for i, (idx, row) in enumerate(top_20_result.iterrows(), 1):
            code = str(row['Code']).zfill(6)
            name = row['Name']
            score = int(row['Trend_Score'])
            print(f"{i:2d}. {name} ({code}) - 트렌드 점수: {score}")
            
        print("\n※ 판다스 데이터프레임 결과물:")
        display(top_20_result)

KRX 상장 종목 리스트를 가져오는 중...
구글 트렌드 데이터를 가져오는 중... (기간: 2025-05-01 ~ 2025-05-31)
✅ 트렌드 연관 검색어 56개 추출 성공.
KRX 종목과 구글 트렌드 검색어를 매칭하는 중...

🏆 구글 트렌드 최다 언급 KRX 상위 20종목 (2025-05-01 ~ 2025-05-31)
 1. 이닉스 (452400) - 트렌드 점수: 14
 2. 한화 (000880) - 트렌드 점수: 8

※ 판다스 데이터프레임 결과물:


,Name,Trend_Score,Code
0,이닉스,14,452400
1,한화,8,000880
